In [ ]:
using Plots
using Revise
using Apr28Project
using DifferentialEquations

# Solve the Advection equation
$$
u_t + c u_x = f(x,t)
$$
with boundary condtion $u(a,t) = g(t)$. This solves it in the matrix free formulation.

In [ ]:
x = LinRange(-5,5, 51)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

# store the parameters in a tuple structure:
p = (c, dx, x, g, f);

In [ ]:
prob = ODEProblem(advection_mf!, uinit, tspan, p);
# sol = solve(prob);
dt = 0.01;
sol = solve(prob);

In [ ]:
sol.alg

We want to visualize a time dependent solution as an animation:

In [ ]:
t_plt = LinRange(tspan[1], tspan[2], 101);
anim = @animate for i in 1:length(t_plt)
    plot(x, sol(t_plt[i]), title = "t = $(round(t_plt[i], digits=2))", ylim=(-1,1),label="")
    xlabel!("x")
    ylabel!("u(x,t)")
end

In [ ]:
gif(anim, "advection.gif", fps=3)

# Dense Matrix Version
This version explicitly constructs a dense matrix representation of the backwards difference operator.

In [ ]:
x = LinRange(-5,5, 51)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

nx = length(x);

Dx = backward_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p = (c, dx, x, g, f, Dx);

In [ ]:
prob = ODEProblem(advection!, uinit, tspan, p);
sol = solve(prob);

In [ ]:
t_plt = LinRange(tspan[1], tspan[2], 101);
anim = @animate for i in 1:length(t_plt)
    plot(x, sol(t_plt[i]), title = "t = $(round(t_plt[i], digits=2))", ylim=(-1,1),label="")
    xlabel!("x")
    ylabel!("u(x,t)")
end
gif(anim, "advection.gif", fps=3)

# Timing Comparisons Between Free and the Dense Version

In [ ]:
using BenchmarkTools

In [ ]:
# set a common problem
nx = 100;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

Dx = backward_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p_mf = (c, dx, x, g, f);
p_dense = (c, dx, x, g, f, Dx);

# prob_mf = ODEProblem(advection_mf!, uinit, tspan, p_mf);
# prob_dense = ODEProblem(advection!, uinit, tspan, p_dense);

du = similar(uinit);
@show nx;
@btime advection_mf!($(du), $(uinit), $(p_mf), 0.0);
@btime advection!($(du), $(uinit), $(p_dense), 0.0);

In [ ]:
# set a common problem
nx = 200;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

Dx = backward_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p_mf = (c, dx, x, g, f);
p_dense = (c, dx, x, g, f, Dx);

# prob_mf = ODEProblem(advection_mf!, uinit, tspan, p_mf);
# prob_dense = ODEProblem(advection!, uinit, tspan, p_dense);

du = similar(uinit);
@show nx;
@btime advection_mf!($(du), $(uinit), $(p_mf), 0.0);
@btime advection!($(du), $(uinit), $(p_dense), 0.0);

In [ ]:
# set a common problem
nx = 400;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

Dx = backward_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p_mf = (c, dx, x, g, f);
p_dense = (c, dx, x, g, f, Dx);

# prob_mf = ODEProblem(advection_mf!, uinit, tspan, p_mf);
# prob_dense = ODEProblem(advection!, uinit, tspan, p_dense);

du = similar(uinit);
@show nx;
@btime advection_mf!($(du), $(uinit), $(p_mf), 0.0);
@btime advection!($(du), $(uinit), $(p_dense), 0.0);


Dense is 300 times slower than matrix free.

In [ ]:
Dx

# Sparse Formulation
This constructs a sparse representation of the backwards difference operator.

In [ ]:
nx = 800; # compare 50, 100, 200, 400,

# set a common problem
x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 10.0);

Dx = sparse_backward_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p_mf = (c, dx, x, g, f);
p_sparse = (c, dx, x, g, f, Dx);

du = similar(uinit);
@show nx;
@btime advection_mf!($(du), $(uinit), $(p_mf), 0.0);
@btime advection!($(du), $(uinit), $(p_sparse), 0.0);

Sparse is slower than matrix free, but does scale linearly with problem size.

In [ ]:
# set a common problem
nx = 50;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there

Dx = sparse_backward_difference_matrix(nx, dx);
Dx

See the desne version:

In [ ]:
Matrix(Dx)

In [ ]:
Dx.nzval